# Question 1: Context Engineering

Try the following experiment:

1. Open ChatGPT in a private browser window: https://chatgpt.com
2. Enter this prompt: "Create a Kestra flow that loads NYC taxi data from CSV to BigQuery"
3. Then, use Kestra's AI Copilot with the same prompt

After trying the same prompt in ChatGPT vs Kestra's AI Copilot, what is the primary reason AI Copilot generates better Kestra flows?

- AI Copilot uses a more powerful model
- AI Copilot has access to current Kestra plugin documentation
- AI Copilot uses more tokens
- AI Copilot has internet access


### ChatGPT:

id: nyc_taxi_csv_to_bigquery
namespace: company.analytics

inputs:
  - id: csv_url
    type: STRING
    defaults: "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-01.csv"

  - id: project_id
    type: STRING
    defaults: "my-gcp-project"

  - id: dataset
    type: STRING
    defaults: "nyc_taxi"

  - id: table
    type: STRING
    defaults: "yellow_tripdata"

tasks:
  - id: download_csv
    type: io.kestra.plugin.core.http.Download
    uri: "{{ inputs.csv_url }}"

  - id: load_to_bigquery
    type: io.kestra.plugin.gcp.bigquery.Load
    from: "{{ outputs.download_csv.uri }}"
    destinationTable: "{{ inputs.project_id }}.{{ inputs.dataset }}.{{ inputs.table }}"
    format: CSV
    autodetect: true
    writeDisposition: WRITE_TRUNCATE
    createDisposition: CREATE_IF_NEEDED
    csvOptions:
      skipLeadingRows: 1
      fieldDelimiter: ","
      allowQuotedNewLines: true

### AI Copilot:

id: wasp_869391
namespace: company.team

tasks:
  - id: download_nyc_taxi_data
    type: io.kestra.plugin.core.http.Download
    uri: https://raw.githubusercontent.com/plotly/datasets/master/taxi_traffic.csv
    saveAs: nyc_taxi_data.csv
  - id: load_to_bigquery
    type: io.kestra.plugin.gcp.bigquery.Load
    from: "{{ outputs.download_nyc_taxi_data.uri }}"
    destinationTable: "my_project.my_dataset.nyc_taxi_data"
    format: CSV
    csvOptions:
      skipLeadingRows: 1

### Answer: AI Copilot has access to current Kestra plugin documentation

# Question 2: RAG vs No RAG


Run both `1_chat_without_rag.yaml` and `2_chat_with_rag.yaml` in the Kestra UI. Read the execution logs for each.

The non-RAG response about Kestra 1.1 features is best described as:

- Accurate and specific, matching the actual release notes
- Vague, generic, or fabricated — the model guesses from training data
- Empty — the model refuses to answer without context
- Identical to the RAG version



INFO 2026-07-21T17:24:54.803374Z ❌ Response WITHOUT RAG (no retrieved context):
Kestra 1.1 introduced several exciting features, significantly enhancing its capabilities and user experience. Here are at least 5 major features with brief descriptions:

1.  **Declarative Input and Output (I/O) with Data Type Validation:**
    *   **Description:** This feature allows users to define expected inputs and outputs for flows and tasks using a declarative syntax. It supports specifying data types (e.g., string, integer, boolean, object, array) and even provides automatic validation. If an input or output doesn't conform to the defined type, Kestra will flag it, preventing common errors and improving the robustness of workflows. This makes flows more predictable, easier to debug, and more maintainable.

2.  **Plugin Bundles:**
    *   **Description:** Instead of managing individual plugins, Kestra 1.1 introduced the concept of "bundles." These are pre-packaged collections of related plugins (e.g., all AWS plugins, all GCP plugins, or specific utility plugins) that can be installed and managed as a single unit. This simplifies plugin management, reduces the configuration overhead for common sets of integrations, and ensures compatibility between bundled plugins.

3.  **Advanced Retry Mechanisms:**
    *   **Description:** Kestra 1.1 brought more sophisticated retry strategies for tasks. Beyond simple fixed retries, it introduced concepts like exponential backoff and custom retry conditions. This allows users to configure retries that are more resilient to transient failures, such as network timeouts or temporary service unavailability, by waiting longer between attempts or retrying only on specific error types, thereby increasing the reliability of complex workflows.

4.  **Flow Template & Blueprint Engine:**
    *   **Description:** This feature allows users to create reusable flow templates and blueprints. Instead of building similar flows from scratch repeatedly, users can define a generic template with placeholders and then instantiate it multiple times with different parameters. This significantly reduces duplication, promotes standardization across an organization, and accelerates the development of new workflows by providing a library of pre-built, tested patterns.

5.  **Enhanced Task & Flow Monitoring in UI:**
    *   **Description:** The Kestra UI received significant improvements in monitoring capabilities. This included more detailed views of task and flow execution, better logging aggregation, and potentially more granular status indicators. These enhancements make it easier for users to track the progress of their workflows, diagnose issues quickly, and gain deeper insights into the performance and health of their Kestra instances. While specific metrics might vary, the overall goal was to provide a richer, more intuitive monitoring experience.

These features collectively made Kestra 1.1 a more powerful, developer-friendly, and robust platform for building and managing complex data orchestration workflows.

🤔 Did you notice that this response seems to be:
- Incorrect?
- Vague/generic?
- Listing features that haven't been added in exactly this version but rather a long time ago?

👉 This is why context matters! Run `2_chat_with_rag.yaml` to see the accurate, context-grounded response.


INFO 2026-07-21T17:32:12.111180Z ✅ RAG Response (with retrieved context):
Kestra 1.1 introduced several major features, including:

1.  **New Filters**: The UI filters were completely redesigned for improved usability, offering explicit filter options, a single-click reset, the ability to save frequently used filter combinations, and customizable table columns.
2.  **No-Code Dashboard Editor**: This feature allows users to create and edit custom dashboards using a no-code, multi-panel editor directly from the UI, similar to the existing no-code flow editor.
3.  **Human Task**: For Enterprise Edition users, this feature enables human-in-the-loop workflows, allowing executions to pause and require manual approval from specific users or groups before proceeding.
4.  **Multi-Agent AI Systems**: AI agents can now use other AI agents as tools, facilitating sophisticated multi-agent orchestration workflows where a primary agent can delegate subtasks to specialized expert agents.
5.  **Fix with AI**: When tasks fail, Kestra 1.1 provides AI-powered suggestions to help users quickly diagnose and resolve issues, speeding up troubleshooting.
6.  **Dozens of New Plugins**: The release included a wide array of new community-driven plugins, expanding integrations across various categories such as Data & Database (e.g., Liquibase, dlt), SaaS & API (e.g., Airtable, Stripe, Shopify), Cloud & Infrastructure (e.g., Dataform, AWS CloudWatch), and AI Model Providers (e.g., OCI GenAI, Cloudflare Workers AI).

🎉 Note that this response is detailed, accurate, and grounded in the actual release documentation. Compare this with the output from 1_chat_without_rag.yaml!


## Answer: Vague, generic, or fabricated — the model guesses from training data


# Question 3: Token usage — short summary

Run `4_simple_agent.yaml` with `summary_length = short` (leave the other inputs as defaults).

Open the execution logs and find the token usage logged by the `log_token_usage` task.

What is the approximate **output** token count for `multilingual_agent`?

- 5-15 tokens
- 60-100 tokens
- 200-400 tokens
- 500+ tokens



INFO 2026-07-21T17:36:29.343702Z 📊 Token Usage Summary:

Multilingual Agent:
- Input tokens: 282
- Output tokens: 72
- Total tokens: 354

English Brevity Agent:
- Input tokens: 87
- Output tokens: 43
- Total tokens: 130

💡 Tip: Monitor token usage to understand costs and optimize prompts!

## Answer: 72 tokens, i.e. 60-100 tokens

# Question 4: Token usage — long summary



Run `4_simple_agent.yaml` again with `summary_length = long`.

Compare the `multilingual_agent` output token count to your result from Question 3. Roughly how many times more output tokens does the long summary use?

- About the same (within 20%)
- 2-5x more
- 10-20x more
- 50x more


INFO 2026-07-21T17:37:36.508748Z 📊 Token Usage Summary:

Multilingual Agent:
- Input tokens: 282
- Output tokens: 162
- Total tokens: 444

English Brevity Agent:
- Input tokens: 177
- Output tokens: 40
- Total tokens: 217

💡 Tip: Monitor token usage to understand costs and optimize prompts!

## Answer: 162 tokens vs 72 tokens from Q3, so 90 tokens more, i.e. 2-5x more

# Question 5: Modifying a flow

Open `4_simple_agent.yaml` in the Kestra flow editor. Find the `english_brevity` task and change its prompt from asking for exactly **1 sentence** to asking for exactly **3 sentences**.

Save the flow, then run it with `summary_length = long`.

Compare the `english_brevity` output token count to the original 1-sentence version (also with `summary_length = long`). How do they compare?

- About the same (within 20%)
- 2-4x more
- 5-10x more
- 10x+ more



INFO 2026-07-21T17:39:48.493119Z 📊 Token Usage Summary:

Multilingual Agent:
- Input tokens: 282
- Output tokens: 179
- Total tokens: 461

English Brevity Agent:
- Input tokens: 194
- Output tokens: 85
- Total tokens: 279

💡 Tip: Monitor token usage to understand costs and optimize prompts!

## Answer: Q4 had 40 tokens in English Brevity, while Q5 has 85 tokens, so 2-4x more

# Question 6: Best Practices

Based on what you learned in this module, for production workflows requiring deterministic, repeatable results with strict compliance requirements (e.g., financial reporting, workflows in highly regulated industries), which approach is most appropriate?

- Always use AI agents for maximum flexibility and adaptation
- Use traditional task-based workflows for predictability and auditability
- Use only RAG without agents for better performance
- Use web search tools exclusively to ensure current data

## Answer: Use traditional task-based workflows for predictability and auditability